In [ ]:
from sctoolbox.utils.jupyter import bgcolor, _compare_version

# change the background of input cells
bgcolor("PowderBlue", select=[3, 5, 8, 10, 13])

nb_name = "Hotspot.ipynb"

_compare_version(nb_name)

# Hotspot
<hr style="border:2px solid black"> </hr>

## 1 - Description

# Hotspot Analysis

Hotspot identifies genes with informative local expression structure in single-cell data by computing gene-wise autocorrelations on a cell neighborhood graph. Based on these genes, local gene-gene correlations can be calculated and grouped into modules representing putative gene programs.
Hotspot documentation: https://hotspot.readthedocs.io/en/latest/

The aim of this notebook is to detect locally variable genes, calculate local gene-gene correlations, and group correlated genes into modules. These modules can represent putative gene programs that are active in specific cell states, developmental stages, or cell populations.

The main outputs of this notebook are Hotspot autocorrelation results, selected informative genes, local correlation matrices, gene modules, module scores per cell, summary tables, and visualizations such as UMAP plots and module-level summaries.

## Prerequisites

Requires a preprocessed AnnData object with raw counts stored in a layer, usually `adata.layers["raw"]`, and relevant metadata in `adata.obs` for plotting or subsetting.

This notebook requires a low-dimensional representation of the data to construct the cell neighborhood graph used by Hotspot. PCA and Harmony-corrected PCA can be computed within this notebook if needed. Other latent representations, such as scVI or Scanorama, must already be present in `adata.obsm` if they should be used for Hotspot.

Move this notebook into the notebook folder of the respective analysis, for example `rna_analysis/notebooks/`, before using it.


_______

## 2 - Setup

In [ ]:
import sctoolbox
import sctoolbox.utils as utils
import sctoolbox.tools.hotspot as hotspot
import sctoolbox.plotting as pl

import pandas as pd
from IPython.display import display
from sctoolbox import settings

sctoolbox.settings.settings_from_config("config.yaml", key="hotspot")

with pd.option_context("display.max.rows", None, "display.max_colwidth", None):
    display(sctoolbox.utils.general.get_version_report(report="versions.yml"))

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Set inpput anndata
clustered_adata = "anndata_4.h5ad"

_________

## 3 - Load anndata

In [ ]:
adata = utils.adata.load_h5ad(clustered_adata)

with pd.option_context("display.max.rows", 5, "display.max.columns", None):
    display(adata)
    display(adata.obs)
    display(adata.var)

## 4 -  Plot UMAP
<hr style="border:2px solid black"> </hr>

This section visualizes the full AnnData object before optional subsetting and Hotspot analysis. The global UMAP provides an overview of the dataset structure and shows how cells are distributed according to the selected condition column.

This helps users inspect sample, condition, batch, or cell population patterns before deciding whether to run Hotspot on the full dataset or on a biologically relevant subset. The UMAP is used for exploratory visualization and does not determine the latent representation used by Hotspot later in the notebook.


<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Global UMAP overview before subsetting

# Column in adata.obs used to color the UMAP plot.
condition_column = "sample"

# Low-dimensional representation used to compute the UMAP if needed.
# Common options are "X_pca", "X_pca_harmony", "X_scVI", or "X_scanorama".
global_umap_use_rep = "X_pca"

# If True, recompute the UMAP even if adata.obsm["X_umap"] already exists.
global_force_recompute_umap = False

# If True, recompute the representation used for UMAP, e.g. PCA or Harmony.
# This only matters if the selected representation is missing or should be overwritten.
global_force_recompute_rep = False

# Shared UMAP computation settings.
# These parameters are passed to the UMAP computation step.
umap_kwargs = {
    "min_dist": 0.5,        # Controls how tightly cells are packed together.
    "spread": 1.0,          # Controls the overall spread of the UMAP layout.
    "n_components": 2,      # Number of UMAP dimensions to compute.
    "random_state": 0,      # Random seed for reproducible UMAP results.
}

# Shared plotting settings for UMAP visualizations.
# These parameters are passed to pl.embedding.plot_embedding().
umap_plot_kwargs = {
    "method": "umap",       # Plot the UMAP embedding stored in adata.obsm["X_umap"].
    "color": condition_column,  # Color cells by the selected metadata column.
    "ncols": 3,             # Number of plot columns if multiple colors are shown.
    "rasterize": True,      # Rasterize points when saving to reduce file size.
    "title": f"UMAP\ncolored by {condition_column}",  # Plot title.
}

In [ ]:
hotspot.ensure_umap_for_visualization(
    adata,
    use_rep=global_umap_use_rep,
    batch_key=condition_column,
    is_subset=False,
    recompute_if_subset=False,
    force_recompute_umap=global_force_recompute_umap,
    force_recompute_rep=global_force_recompute_rep,
    neighbors_key="neighbors_global_umap",
    **umap_kwargs,
)

_ = pl.embedding.plot_embedding(
    adata,
    save="hotspot_subset_umap.pdf",
    report="hotspot_subset_umap.png",
    **umap_plot_kwargs,
)

__________

## 5 - Subset data (optional)
<hr style="border:2px solid black"> </hr>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# ============================================================
# Optional subsetting before running Hotspot
# If None the Anndata won't be changed
# Example:
# subset_filters = {
#     "sample": [3, 6],
#     "subcelltype": ["A+", "A-"],
# }
# ============================================================

subset_filters = {}
subset_copy = True
drop_unused_categories = False

In [ ]:
# Subset the celltype an early stage for example
adata_used = hotspot.subset_adata(
        adata=adata,
        subset_filters=subset_filters,
        copy=subset_copy,
        drop_unused_categories=drop_unused_categories,
    )
adata_used

In [ ]:
# Set subset true
subset_was_applied = hotspot.was_subsetted(
    original_adata=adata,
    current_adata=adata_used,
)

print(f"Subset was applied: {subset_was_applied}")

## 6 - Filter genes
<hr style="border:2px solid black"> </hr>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Gene detection filtering settings
GENE_DETECTION_FILTERING = {
    # Layer used for detection count.
    # Usually raw counts are preferred, e.g. "raw", "counts", or None for adata.X
    "layer_key": "raw",

    # Minimum number of cells in which a gene must be detected
    "min_cells": 50,

    # Only inspect thresholds, useful for deciding min_cells
    "thresholds_to_show": [10, 20, 30, 50, 100],

    # If False, only show statistics and plot.
    # If True, actually remove genes detected in fewer than min_cells cells.
    "filter_genes": False,
}

In [ ]:
adata_filtered, gene_detection, gene_filter_summary = hotspot.inspect_and_filter_genes_by_cell_detection(
    adata_used,
    **GENE_DETECTION_FILTERING,
)

## 7 - Select and calculate Hotspot latent key for hotspot
<hr style="border:2px solid black"> </hr>

### Parameter Overview

The following parameters define which metadata columns, latent representation, and dimensionality reduction settings are used for Hotspot and visualization.
| Parameter | Description | Options / Example |
|:----------|:------------|:------------------|
| `condition_column`                                               | Column in `adata.obs` used for plotting and coloring cells in visualizations such as UMAP.                                                                                                     | String, e.g. `"sample"`, `"cell_type"`, `"condition"`                       |
| `batch_key`                                                      | Column in `adata.obs` used for Harmony batch correction. Set to `None` if no batch correction should be applied.                                                                               | `None` or String, e.g. `"sample"`, `"batch"`                                |
| `HOTSPOT_REPRESENTATION["preferred_rep"]`                        | Defines which low-dimensional representation should be used for Hotspot. If set to `"auto"`, the notebook selects the best available representation based on the priority list.                | `"auto"`, `"X_pca"`, `"X_pca_harmony"`, `"X_scVI"`, `"X_scanorama"`         |
| `HOTSPOT_REPRESENTATION["priority"]`                             | Ordered list of representations used when `preferred_rep="auto"`. The first available or computable representation is selected.                                                                | List of strings, e.g. `["X_pca_harmony", "X_scVI", "X_scanorama", "X_pca"]` |
| `HOTSPOT_REPRESENTATION["compute_pca_if_missing"]`               | Whether PCA should be computed if no suitable representation is available.                                                                                                                     | `True` or `False`                                                           |
| `HOTSPOT_REPRESENTATION["compute_harmony_if_missing"]`           | Whether Harmony-corrected PCA should be computed if Harmony is selected or prioritized and a valid `batch_key` is available.                                                                   | `True` or `False`                                                           |
| `HOTSPOT_REPRESENTATION["recompute_if_subset"]`                  | Whether PCA or Harmony should be recomputed after the AnnData object was subsetted. This is recommended when Hotspot is run on a specific subset of cells.                                     | `True` or `False`                                                           |
| `HOTSPOT_REPRESENTATION["force_recompute_pca"]`                  | Whether an existing PCA representation should be overwritten and recomputed.                                                                                                                   | `True` or `False`                                                           |
| `HOTSPOT_REPRESENTATION["force_recompute_harmony"]`              | Whether an existing Harmony representation should be overwritten and recomputed.                                                                                                               | `True` or `False`                                                           |
| `HOTSPOT_REPRESENTATION["allow_existing_external_after_subset"]` | Whether existing external representations such as scVI or Scanorama may still be used after subsetting. These representations are not recomputed automatically by this notebook.               | `True` or `False`                                                           |
| `PCA_SETTINGS["pca_key"]`                                        | Key under which the PCA representation is stored in `adata.obsm`.                                                                                                                              | String, usually `"X_pca"`                                                   |
| `PCA_SETTINGS["n_comps"]`                                        | Number of principal components to compute.                                                                                                                                                     | Integer, e.g. `50`                                                          |
| `PCA_SETTINGS["use_highly_variable"]`                            | Whether PCA should be computed only on highly variable genes. This requires highly variable genes to be annotated in `adata.var`.                                                              | `True` or `False`                                                           |
| `PCA_SETTINGS["layer"]`                                          | Layer used for PCA computation. If `None`, `adata.X` is used.                                                                                                                                  | `None` or String, e.g. `"raw"`                                              |
| `PCA_SETTINGS["svd_solver"]`                                     | SVD solver passed to Scanpy PCA. If `None`, Scanpy uses its default behavior.                                                                                                                  | `None` or String                                                            |
| `PCA_SETTINGS["random_state"]`                                   | Random seed used for reproducible PCA results.                                                                                                                                                 | Integer, e.g. `0`                                                           |
| `HARMONY_SETTINGS["basis"]`                                      | Input representation used for Harmony correction.                                                                                                                                              | String, usually `"X_pca"`                                                   |
| `HARMONY_SETTINGS["adjusted_basis"]`                             | Key under which the Harmony-corrected representation is stored in `adata.obsm`.                                                                                                                | String, usually `"X_pca_harmony"`                                           |
| `HARMONY_SETTINGS["batch_key"]`                                  | Metadata column used by Harmony to correct for batch effects. Usually set from `batch_key`.                                                                                                    | `None` or String, e.g. `"sample"`                                           |
| `subset_umap_use_rep`                                            | Low-dimensional representation used to compute the subset UMAP for visualization. This setting affects the UMAP plot only and does not automatically change the Hotspot latent representation. | `"X_pca"`, `"X_pca_harmony"`, `"X_scVI"`, `"X_scanorama"`                   |


<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Metadata columns
condition_column = "sample"      # for plotting / coloring
batch_key = "sample"
# Hotspot latent representation settings
HOTSPOT_REPRESENTATION = {
    # Options:
    # "auto"           -> use next available representation from priority
    # "X_pca"          -> compute/use PCA
    # "X_pca_harmony"  -> compute/use Harmony-corrected PCA
    # "X_scVI"         -> use existing scVI representation if available
    # "X_scanorama"    -> use existing Scanorama representation if available
    "preferred_rep": "auto",

    # Priority if preferred_rep == "auto"
    "priority": [
        "X_pca_harmony",
        "X_scVI",
        "X_scanorama",
        "X_pca",
    ],

    # If no suitable representation exists, compute PCA
    "compute_pca_if_missing": True,

    # If batch_key is available and Harmony is wanted
    "compute_harmony_if_missing": True,

    # If the object was subsetted, recompute PCA/Harmony
    "recompute_if_subset": True,

    # Do not overwrite existing embeddings unless explicitly wanted
    "force_recompute_pca": False,
    "force_recompute_harmony": False,

    # External embeddings cannot be recomputed automatically.
    # If False, X_scVI / X_scanorama are skipped after subsetting.
    # If True, existing subsetted X_scVI / X_scanorama can still be used.
    "allow_existing_external_after_subset": False,
}

# PCA settings
PCA_SETTINGS = {
    "pca_key": "X_pca",
    "n_comps": 50,
    "use_highly_variable": False,
    "layer": None,
    "svd_solver": None,
    "random_state": 0,
}

# Harmony settings
HARMONY_SETTINGS = {
    "basis": "X_pca",
    "adjusted_basis": "X_pca_harmony",
    "batch_key": batch_key,
}

# UMAP visualization setting after optional subsetting
subset_umap_use_rep = "X_pca"

In [ ]:
hotspot_latent_key = hotspot.ensure_hotspot_latent_representation(
    adata=adata_filtered,
    representation_settings=HOTSPOT_REPRESENTATION,
    pca_settings=PCA_SETTINGS,
    harmony_settings=HARMONY_SETTINGS,
    is_subset=subset_was_applied,
)

print(f"Selected Hotspot latent representation: {hotspot_latent_key}")

## 8 - Calculate UMAP after subsetting
<hr style="border:2px solid black"> </hr>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Whether to recompute the UMAP representation after subsetting.
# If True, PCA/Harmony and UMAP can be recomputed for the subset.
subset_recompute_if_subset = True

# If True, recompute the UMAP even if adata_filtered.obsm["X_umap"] already exists.
subset_force_recompute_umap = False

# If True, recompute the representation used for UMAP, e.g. PCA or Harmony.
subset_force_recompute_rep = False

# Key used to store the neighbor graph for the subset UMAP.
subset_neighbors_key = "neighbors_hotspot_umap"

# Shared plotting settings for the subset UMAP.
# These settings are passed to pl.embedding.plot_embedding().
umap_plot_kwargs = {
    "method": "umap",  # Plot the UMAP embedding stored in adata.obsm["X_umap"].
    "color": condition_column,  # Color cells by the selected metadata column.
    "ncols": 3,  # Number of plot columns if multiple colors are shown.
    "rasterize": True,  # Rasterize points when saving to reduce file size.
    "title": f"UMAP after subsetting\ncolored by {condition_column}",
}

In [ ]:
hotspot.ensure_umap_for_visualization(
    adata_filtered,
    use_rep=subset_umap_use_rep,
    batch_key=batch_key,
    is_subset=subset_was_applied,
    recompute_if_subset=subset_recompute_if_subset,
    force_recompute_umap=subset_force_recompute_umap,
    force_recompute_rep=subset_force_recompute_rep,
    neighbors_key=subset_neighbors_key,
    use_highly_variable=PCA_SETTINGS["use_highly_variable"],
    **umap_kwargs,
)

_ = pl.embedding.plot_embedding(
    adata_filtered,
    save="hotspot_subset_umap.pdf",
    report="hotspot_subset_umap.png",
    **umap_plot_kwargs,
)

## 9 - Hotspot Run
<hr style="border:2px solid black"> </hr>

### Parameter Overview

The following parameters control the Hotspot input data, neighborhood graph construction, gene selection, module detection, and output handling.

| Parameter                    | Description                                                                                                                                                                     | Options / Example                                                               |
| ---------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------- |
| `output_prefix`              | Prefix used for saved result files.                                                                                                                                             | String, e.g. `"hotspot"`                                                        |
| `adata_output`               | File name for saving the AnnData object after adding Hotspot results.                                                                                                           | String, e.g. `"hotspot.h5ad"`                                                   |
| `layer_key`                  | Layer used as count input for Hotspot. This should contain raw or count-like expression values.                                                                                 | String, e.g. `"raw"`                                                            |
| `latent_obsm_key`            | Low-dimensional representation from `adata.obsm` used by Hotspot to construct the cell neighborhood graph. This is usually selected in the previous latent representation step. | `hotspot_latent_key`, `"X_pca"`, `"X_pca_harmony"`, `"X_scVI"`, `"X_scanorama"` |
| `umi_counts_obs_key`         | Column in `adata.obs` containing total UMI/count values per cell. If missing, it can be computed from the selected layer.                                                       | String, e.g. `"total_counts"`                                                   |
| `model`                      | Statistical model used by Hotspot for gene expression. For count-based scRNA-seq data, `"danb"` is commonly used.                                                               | `"danb"`, `"bernoulli"`, `"normal"`                                             |
| `n_neighbors`                | Number of neighboring cells used to construct the Hotspot KNN graph.                                                                                                            | Integer, e.g. `30`                                                              |
| `weighted_graph`             | Whether to use a weighted neighborhood graph in Hotspot.                                                                                                                        | `True` or `False`                                                               |
| `jobs`                       | Number of parallel jobs used for Hotspot computations.                                                                                                                          | Integer, e.g. `8`                                                               |
| `gene_selection_method`      | Strategy used to select genes for local correlation analysis.                                                                                                                   | `"fdr"` or `"top_n_z"`                                                          |
| `autocorr_fdr_threshold`     | FDR threshold for selecting informative genes when `gene_selection_method="fdr"`. Ignored when using `"top_n_z"`.                                                               | Float, e.g. `0.05`                                                              |
| `top_n`                      | Number of top genes selected by Hotspot Z-score when `gene_selection_method="top_n_z"`. Ignored when using `"fdr"`.                                                             | Integer, e.g. `500`                                                             |
| `min_gene_threshold`         | Minimum number of genes required for a Hotspot module. Smaller values allow smaller modules.                                                                                    | Integer, e.g. `20`                                                              |
| `core_only`                  | Whether to keep only core genes assigned to Hotspot modules.                                                                                                                    | `True` or `False`                                                               |
| `module_fdr_threshold`       | FDR threshold used during Hotspot module creation.                                                                                                                              | Float, e.g. `0.05`                                                              |
| `compute_local_correlations` | Whether to compute local gene-gene correlations for the selected genes. Required for module detection.                                                                          | `True` or `False`                                                               |
| `compute_modules`            | Whether to group locally correlated genes into Hotspot modules.                                                                                                                 | `True` or `False`                                                               |
| `compute_module_scores`      | Whether to calculate per-cell module scores for the detected modules.                                                                                                           | `True` or `False`                                                               |
| `attach_results_to_adata`    | Whether to add Hotspot results such as module scores and gene-level statistics back to the AnnData object.                                                                      | `True` or `False`                                                               |
| `module_prefix`              | Prefix used for module score columns added to `adata.obs`.                                                                                                                      | String, e.g. `"Hotspot module"`                                                 |
| `overwrite_attached_results` | Whether existing Hotspot result columns in `adata` should be overwritten.                                                                                                       | `True` or `False`                                                               |


<h1><center>⬐ Fill in input data here ⬎</center></h1>

### 9.1 - input hotspot parameters

In [ ]:
# Input / output
output_prefix = "hotspot"
adata_output = "anndata_hotspot.h5ad"

# Hotspot input settings
layer_key = "raw"
latent_obsm_key = hotspot_latent_key
umi_counts_obs_key = "total_counts"
model = "danb"

# Graph settings
n_neighbors = 30
weighted_graph = False
jobs = settings.threads

# Gene selection
gene_selection_method = "top_n_z"   # available: "fdr", "top_n_z"
autocorr_fdr_threshold = 0.05
top_n = 500

# Module settings
min_gene_threshold = 20
core_only = True
module_fdr_threshold = 0.05

# Outputs
compute_local_correlations = True
compute_modules = True
compute_module_scores = True

# Whether Hotspot results should be added to adata
attach_results_to_adata = True
module_prefix = "Hotspot module"
overwrite_attached_results = True

### 9.2 - Run hotspot

In [ ]:
results = hotspot.run_hotspot(
    adata=adata_filtered,
    layer_key=layer_key,
    latent_obsm_key=latent_obsm_key,
    umi_counts_obs_key=umi_counts_obs_key,
    model=model,
    n_neighbors=n_neighbors,
    weighted_graph=weighted_graph,
    jobs=jobs,
    top_n=top_n,
    gene_selection_method=gene_selection_method,
    autocorr_fdr_threshold=autocorr_fdr_threshold,
    min_gene_threshold=min_gene_threshold,
    core_only=core_only,
    module_fdr_threshold=module_fdr_threshold,
    compute_local_correlations=compute_local_correlations,
    compute_modules=compute_modules,
    compute_module_scores=compute_module_scores,
    attach_results_to_adata=attach_results_to_adata,
    module_prefix=module_prefix,
    overwrite_attached_results=overwrite_attached_results,
)

_________

## 10 - Results
<hr style="border:2px solid black"> </hr>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
fdr_threshold = 0.05
store_local_correlations = False

In [ ]:
summary = hotspot.show_hotspot_results(
    adata=adata_filtered,
    results=results,
    fdr_threshold=fdr_threshold,
    output_prefix=output_prefix,
)

## 11 - Module enrichment
<hr style="border:2px solid black"> </hr>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
gene_sets = ["MSigDB_Hallmark_2020", "Reactome_2022",]
organism = "human"
min_module_size = 5
overwrite = True
inplace = True
save_table = "hotspot_module_enrichment.tsv"

# Plot enrichment
top_n = 5


### 11.1 - Run module enrichment

In [ ]:
hotspot.run_hotspot_module_enrichment(
    adata=adata_filtered,
    results=results,
    gene_sets=gene_sets,
    organism=organism,
    min_module_size=min_module_size,
    overwrite=overwrite,
    inplace=inplace,
    save_table=save_table,
)

### 11.2 - plot module enrichment

In [ ]:
# Plot module enrichment
hotspot.plot_hotspot_module_enrichment(
    adata_filtered,
    top_n=top_n,
    save="hotspot_module_enrichment.pdf",
    report="hotspot_module_enrichment.png"
)

## 12 - Save filtered adata
<hr style="border:2px solid black"> </hr>
Store the final results

In [ ]:
adata_filtered = hotspot.add_hotspot_results_to_adata(
    adata=adata_filtered,
    results=results,
    prefix=output_prefix,
    fdr_threshold=fdr_threshold,
    store_local_correlations=store_local_correlations,
)

In [ ]:
# Saving the data
adata_filtered = hotspot.stringify_adata_keys_for_saving(adata_filtered)
utils.adata.save_h5ad(adata_filtered, adata_output, report=[
        "hotspot_overview.md",
        "hotspot_obs.png",
        "hotspot_var.png"])

In [ ]:
sctoolbox.settings.close_logfile()